# 06 - Ingestão Bronze: Controle de Hash e Ingestão Incremental de Pedidos
**Squad 2 — Real Time for Business | Dupla 2**  
**Integrantes:** Kauan & Leandro  
**Tabelas de Escopo:** `ecommerce_pedidos`  
**Branch:** `feat/squad2/kauan-leandro`  

### Objetivo da Task (Missão 01 — Bronze):
1. **Leitura da Origem:** Ler `ecommerce_pedidos` via Lakehouse Federation (SQL Server).
2. **Controle de Hash:** Calcular hash SHA-256 de cada pedido para detectar duplicatas e alterações.
3. **Ingestão Incremental:** Filtrar apenas pedidos novos ou alterados via anti-join com a tabela de controle.
4. **Persistência Bronze:** Append incremental em `pedidos_filtrados` + MERGE em `controle_pedidos`.
5. **Idempotência:** Reexecução não duplica registros já processados.

## KPIs e Tabelas Geradas

| Tabela UC | Função | Modo |
| --- | --- | --- |
| `workspace.default.controle_pedidos` | Rastreia hash SHA-256 de cada pedido | Upsert (MERGE) |
| `workspace.default.pedidos_filtrados` | Pedidos novos/alterados (camada Bronze) | Append incremental |

## Fluxo de Execução

1. **Setup** — Carrega credenciais do `.env` e testa conectividade ADLS (fallback UC)
2. **Configuração** — Define camadas ADLS (primário) + UC (fallback serverless)
3. **Tabela de controle** — Cria `controle_pedidos` se não existir
4. **Leitura da origem** — Lê `ecommerce_pedidos` via Lakehouse Federation
5. **Cálculo do hash** — SHA-256 de todas as colunas (exceto `id_pedido`)
6. **Comparação com controle** — Anti-join: filtra apenas pedidos novos ou alterados
7. **Persistência** — Append em `pedidos_filtrados` + MERGE em `controle_pedidos`
8. **Resumo** — Total na origem, novos/alterados, já processados, total acumulado

**Arquitetura dual (ADLS + UC fallback):** Tenta acessar o ADLS Gen2 primeiro (original do Kauan). Se o ADLS estiver bloqueado (serverless), faz fallback automático para Unity Catalog + Lakehouse Federation.

In [0]:
%run ./00_setup_config

In [0]:
# Importação das bibliotecas necessárias

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import os

In [0]:
# ============================================================================
# Configuração de camadas: ADLS (primário) + Unity Catalog (fallback serverless)
# ============================================================================

# --- Opção 1: ADLS Gen2 (original do Kauan) ---
storage_account_name = "internshipdatalake"

adls_options = {
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": os.getenv("ADLS_CLIENT_ID"),
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": os.getenv("ADLS_CLIENT_SECRET"),
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": (
        f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token"
    )
}

# --- Opção 2: Unity Catalog + Lakehouse Federation (fallback para serverless) ---
tabela_origem_pedidos = "sqlserver_catalog.squad2.ecommerce_pedidos"
tabela_controle = "workspace.default.controle_pedidos"
tabela_pedidos_filtrados = "workspace.default.pedidos_filtrados"

# --- Teste de conectividade: tenta ADLS primeiro, fallback para UC ---
try:
    spark.read.format("parquet").options(**adls_options).load(
        f"abfss://raw@{storage_account_name}.dfs.core.windows.net/"
    ).limit(1).count()
    usar_adls = True
    print("✅ ADLS acessível — usando camadas no ADLS (original)")
except Exception:
    usar_adls = False
    print("⚠️ ADLS bloqueado (serverless) — usando Unity Catalog + Lakehouse Federation (fallback)")

print(f"Origem pedidos : {tabela_origem_pedidos if not usar_adls else 'ADLS raw/real-time-data/'}")
print(f"Controle       : {tabela_controle if not usar_adls else 'ADLS grupo2/metadata/'}")
print(f"Filtrados      : {tabela_pedidos_filtrados if not usar_adls else 'ADLS grupo2/bronze/'}")

---
## 📋 Tabela de Controle

Cria a tabela `workspace.default.controle_pedidos` que rastreia o hash SHA-256 de cada pedido já processado. Esta tabela é usada no passo de comparação para identificar apenas pedidos novos ou alterados.

In [0]:
# Cria a tabela de controle se não existir

spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.default.controle_pedidos (
        id_pedido           BIGINT,
        hash_registro       STRING,
        dt_processamento    TIMESTAMP
    )
    USING DELTA
    COMMENT 'Controle de estado: rastreia quais pedidos já foram processados (por hash SHA-256).'
""")

print("✅ Tabela de controle verificada/criada: workspace.default.controle_pedidos")

---
## 📖 Leitura da Origem

Lê a tabela `ecommerce_pedidos` da origem (ADLS ou Lakehouse Federation, dependendo da conectividade).

In [0]:
# Lê a tabela de pedidos da origem

if usar_adls:
    # ADLS: lê os arquivos Parquet do container raw
    # (código a definir com o Kauan — depende da estrutura de pastas dele)
    raise NotImplementedError("ADLS: definir leitura dos parquets de ecommerce_pedidos com o Kauan")
else:
    # UC fallback: lê via Lakehouse Federation
    df_pedidos = spark.table(tabela_origem_pedidos)

print(f"📊 Total de pedidos na origem: {df_pedidos.count()}")
display(df_pedidos.limit(5))

---
## 🔐 Cálculo do Hash SHA-256

Calcula o hash SHA-256 de todas as colunas do registro (exceto `id_pedido`), concatenadas com `||`. Este hash identifica se um pedido foi alterado desde o último processamento.

In [0]:
# Calcula o hash SHA-256 de cada registro (todas as colunas exceto id_pedido)

colunas_hash = [c for c in df_pedidos.columns if c != "id_pedido"]

expr_hash = F.sha2(
    F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("NULL")) for c in colunas_hash]),
    256
)

df_com_hash = df_pedidos.withColumn("hash_registro", expr_hash)

print(f"Hash calculado para {len(colunas_hash)} colunas.")
print(f"Colunas: {colunas_hash}")

---
## 🔍 Comparação com Controle

Compara o hash atual com o hash armazenado na tabela de controle:
* **`hash_antigo IS NULL`** → pedido novo (nunca processado)
* **`hash_novo != hash_antigo`** → pedido alterado (reprocessar)
* **`hash_novo == hash_antigo`** → pedido inalterado (ignorar)

In [0]:
# Compara com a tabela de controle para encontrar apenas pedidos novos ou alterados

df_controle = spark.table(tabela_controle)

df_diff = df_com_hash.alias("src").join(
    df_controle.alias("ctl"),
    F.col("src.id_pedido") == F.col("ctl.id_pedido"),
    "left"
).select(
    F.col("src.id_pedido"),
    F.col("src.hash_registro").alias("hash_novo"),
    F.col("ctl.hash_registro").alias("hash_antigo"),
)

# Filtra: hash_antigo IS NULL (pedido novo) OU hash_novo != hash_antigo (pedido alterado)
df_novos = df_diff.filter(
    F.col("hash_antigo").isNull() | (F.col("hash_novo") != F.col("hash_antigo"))
)

total_novos = df_novos.count()
total_origem = df_com_hash.count()

print(f"📊 Total de pedidos na origem     : {total_origem}")
print(f"📊 Pedidos novos ou alterados      : {total_novos}")
print(f"📊 Pedidos já processados (ignorar): {total_origem - total_novos}")

---
## 💾 Persistência Bronze

* **`pedidos_filtrados`** — Append dos pedidos novos/alterados (camada Bronze incremental)
* **`controle_pedidos`** — MERGE (upsert): insere novos, atualiza hash de alterados

In [0]:
# Salva apenas os pedidos novos/alterados e atualiza a tabela de controle

if total_novos > 0:
    # 1. Gravar na tabela de pedidos filtrados (append)
    df_pedidos_novos = df_com_hash.join(df_novos.select("id_pedido"), "id_pedido", "inner")
    
    df_pedidos_novos.write.mode("append").saveAsTable(tabela_pedidos_filtrados)
    print(f"✅ {total_novos} pedidos inseridos em {tabela_pedidos_filtrados}")
    
    # 2. Atualizar a tabela de controle (MERGE: insere novos, atualiza alterados)
    df_atualizacao = df_novos.select(
        F.col("id_pedido"),
        F.col("hash_novo").alias("hash_registro"),
        F.current_timestamp().alias("dt_processamento")
    )
    
    df_atualizacao.createOrReplaceTempView("vw_atualizacao_pedidos")
    
    spark.sql("""
        MERGE INTO workspace.default.controle_pedidos AS ctl
        USING vw_atualizacao_pedidos AS src
        ON ctl.id_pedido = src.id_pedido
        WHEN MATCHED THEN UPDATE SET
            ctl.hash_registro = src.hash_registro,
            ctl.dt_processamento = src.dt_processamento
        WHEN NOT MATCHED THEN INSERT (id_pedido, hash_registro, dt_processamento)
        VALUES (src.id_pedido, src.hash_registro, src.dt_processamento)
    """)
    
    print(f"✅ Tabela de controle atualizada: {tabela_controle}")
else:
    print("Nenhum pedido novo ou alterado. Nada a processar.")

---
## 📊 Resumo Geral

Consolida os resultados da ingestão Bronze: total na origem, novos/alterados, já processados e total acumulado nas tabelas de controle e filtrados.

In [0]:
# Resumo final

print("=== RESUMO DA INGESTÃO BRONZE — ecommerce_pedidos ===")
print(f"Origem             : {tabela_origem_pedidos if not usar_adls else 'ADLS'}")
print(f"Total na origem    : {total_origem}")
print(f"Novos/alterados    : {total_novos}")
print(f"Já processados     : {total_origem - total_novos}")
print(f"Tabela filtrados   : {tabela_pedidos_filtrados}")
print(f"Tabela controle    : {tabela_controle}")

# Mostra o total acumulado na tabela de filtrados
total_acumulado = spark.table(tabela_pedidos_filtrados).count()
print(f"\nTotal acumulado em {tabela_pedidos_filtrados}: {total_acumulado}")

# Mostra o total na tabela de controle
total_controle = spark.table(tabela_controle).count()
print(f"Total em {tabela_controle}: {total_controle}")